# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeyeshbaral/ShreeyeshAssignment1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 4 — CTR / Engagement Opportunity Scoring.**

This notebook builds the transparent rule-based baseline that any Week-5 model must beat.
It checks two signals with bucket tables, encodes one hand-written rule with a score,
reason code, and action label, writes the ranked queue to CSV, and reviews the top 10
with a skeptic's eye.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `building-baselines` + `flyrank/flyrank-data`.

In [1]:
# ── Section 0: Load data and build the Lane 4 working slice ──────────────
import pandas as pd
import numpy as np
import os, json, pathlib

# Handle both Colab (cloned repo) and local paths
local_path = '../../data/raw/content_refresh_anonymized.csv'
colab_path = '/content/ShreeyeshAssignment1/data/raw/content_refresh_anonymized.csv'

if os.path.exists(local_path):
    csv_path = local_path
elif os.path.exists(colab_path):
    csv_path = colab_path
else:
    raise FileNotFoundError('Could not find content_refresh_anonymized.csv')

df = pd.read_csv(csv_path)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Lane 4 working slice: exclude no-position rows and low-volume noise
lane4 = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()
print(f'Lane 4 working slice: {len(lane4):,} rows  (excluded {len(df) - len(lane4):,})')
print(f'Distinct clients: {lane4["client_id"].nunique()}')

# Build the proxy label: is_under_ctr
tier_median = lane4.groupby('position_tier')['ctr'].median()
lane4['tier_median_ctr'] = lane4['position_tier'].map(tier_median.to_dict())
lane4['ctr_gap'] = lane4['tier_median_ctr'] - lane4['ctr']
lane4['is_under_ctr'] = (lane4['ctr'] < lane4['tier_median_ctr']).astype(int)

base_rate = lane4['is_under_ctr'].mean()
print(f'\nProxy label base rate: {base_rate:.1%} of pages are under their tier median CTR')
print(f'  (under-CTR: {lane4["is_under_ctr"].sum():,}  |  at/above: {(1 - lane4["is_under_ctr"]).sum():,.0f})')

Loaded: 30,000 rows × 44 columns
Lane 4 working slice: 22,006 rows  (excluded 7,994)
Distinct clients: 30

Proxy label base rate: 46.8% of pages are under their tier median CTR
  (under-CTR: 10,307  |  at/above: 11,699)


## 1. My rule and its reasoning — two signal checks first

Before encoding the rule, I check two signals it will lean on. At least one must be
behind a real FlyRank flag from the session.

### Signal A: CTR-vs-Position (flag-linked: `low_ctr_visible_page`)

The FlyRank baseline flags pages with `impressions >= 500`, `avg_position <= 20`, and
`ctr < 0.5%` as `low_ctr_visible_page`. The signal behind this flag is that **CTR
drops with worse position**, but **the spread within each tier is wide** — meaning
position alone doesn't explain the gap, and there's room for other signals.

I'll bucket by `position_tier` and check whether the under-CTR rate varies across tiers.

In [2]:
# ── Signal A: CTR-vs-Position bucket table ─────────────────────────────────
# Flag-linked: behind FlyRank's low_ctr_visible_page flag

tier_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']

rows_a = []
for tier in tier_order:
    sub = lane4[lane4['position_tier'] == tier]
    if len(sub) == 0:
        continue
    rows_a.append({
        'position_tier': tier,
        'n': len(sub),
        'median_ctr': round(sub['ctr'].median(), 3),
        'Q25_ctr': round(sub['ctr'].quantile(0.25), 3),
        'Q75_ctr': round(sub['ctr'].quantile(0.75), 3),
        'pct_under_ctr': round(sub['is_under_ctr'].mean() * 100, 1),
        'median_impressions': int(sub['impressions_90d'].median()),
    })

sig_a = pd.DataFrame(rows_a)
print('Signal A — CTR by position tier (CTR is ×100 percentage: 0.24 = 0.24%)')
print(f'n = {len(lane4):,} total pages in the working slice\n')
print(sig_a.to_string(index=False))

print()
print('Observations:')
print('  • Median CTR falls from 0.19% (top_3) → 0.00% (deep) — position matters.')
print('  • But Q25–Q75 spans 2–4× within every tier — position alone does NOT explain the gap.')
print('  • The under-CTR rate hovers near 50% in every tier (by construction of the median),')
print('    but the SIZE of the gap and the volume behind it differ substantially.')
print()
print('Verdict: CONFIRMED — CTR varies by position, and within-tier spread is wide.')
print('  The low_ctr_visible_page flag is grounded in a real, measurable pattern.')

Signal A — CTR by position tier (CTR is ×100 percentage: 0.24 = 0.24%)
n = 22,006 total pages in the working slice

position_tier    n  median_ctr  Q25_ctr  Q75_ctr  pct_under_ctr  median_impressions
        top_3  533        0.19     0.05     0.48           49.5                2918
       page_1 8633        0.23     0.09     0.46           49.4                2945
     striking 5903        0.15     0.00     0.34           48.6                1388
     page_3_5 6058        0.06     0.00     0.19           48.1                1210
         deep  879        0.00     0.00     0.00            0.0                 426

Observations:
  • Median CTR falls from 0.19% (top_3) → 0.00% (deep) — position matters.
  • But Q25–Q75 spans 2–4× within every tier — position alone does NOT explain the gap.
  • The under-CTR rate hovers near 50% in every tier (by construction of the median),
    but the SIZE of the gap and the volume behind it differ substantially.

Verdict: CONFIRMED — CTR varies by posit

### Signal B: Staleness / days_since_last_update (flag-linked: `stale_visible_page`)

The FlyRank baseline flags pages with `days_since_last_update >= 180` and
`impressions >= 500` as `stale_visible_page`. The hypothesis is that **stale content
has outdated titles and meta descriptions**, leading to lower CTR.

I'll bucket by `freshness_tier` among high-volume pages (≥500 impressions) and check
whether staleness associates with being under-CTR.

In [3]:
# ── Signal B: Staleness bucket table ───────────────────────────────────────
# Flag-linked: behind FlyRank's stale_visible_page flag
# Restrict to pages with >= 500 impressions (so volume noise doesn't dominate)

hv = lane4[lane4['impressions_90d'] >= 500].copy()
print(f'High-volume subset (impressions >= 500): n = {len(hv):,}\n')

freshness_order = ['0-30', '31-90', '91-180', '181+']

rows_b = []
for tier in freshness_order:
    sub = hv[hv['freshness_tier'] == tier]
    if len(sub) == 0:
        continue
    rows_b.append({
        'freshness_tier': tier,
        'n': len(sub),
        'median_ctr': round(sub['ctr'].median(), 3),
        'pct_under_ctr': round(sub['is_under_ctr'].mean() * 100, 1),
        'median_days_since_update': int(sub['days_since_last_update'].median()),
        'median_impressions': int(sub['impressions_90d'].median()),
    })

sig_b = pd.DataFrame(rows_b)
print('Signal B — Under-CTR rate by freshness tier (high-volume pages only)')
print(sig_b.to_string(index=False))

# Also check: within page_1 tier only (controlling for position)
print('\n--- Position-controlled check: page_1 tier only ---')
hv_p1 = hv[hv['position_tier'] == 'page_1']
rows_b2 = []
for tier in freshness_order:
    sub = hv_p1[hv_p1['freshness_tier'] == tier]
    if len(sub) < 10:
        continue
    rows_b2.append({
        'freshness_tier': tier,
        'n': len(sub),
        'median_ctr': round(sub['ctr'].median(), 3),
        'pct_under_ctr': round(sub['is_under_ctr'].mean() * 100, 1),
    })

sig_b2 = pd.DataFrame(rows_b2)
print(f'n = {len(hv_p1):,} page_1 high-volume pages\n')
print(sig_b2.to_string(index=False))

print()
print('Observations:')
print('  • The under-CTR rate does not show a strong monotonic increase with staleness.')
print('  • Even within page_1 (controlling for position), the freshest tier (0-30 days)')
print('    does not consistently outperform stale tiers on CTR.')
print('  • This suggests staleness is a WEAK signal for CTR under-performance, not a strong one.')
print('  • However, stale pages are still worth flagging because they are the most ACTIONABLE —')
print('    updating a stale title is a concrete step an editor can take.')
print()
print('Verdict: MIXED — staleness does not strongly predict under-CTR in this data,')
print('  but stale pages remain the most actionable targets for a content team.')
print('  The stale_visible_page flag is grounded in actionability, not pure signal strength.')

High-volume subset (impressions >= 500): n = 16,726

Signal B — Under-CTR rate by freshness tier (high-volume pages only)
freshness_tier     n  median_ctr  pct_under_ctr  median_days_since_update  median_impressions
          0-30 10063        0.18           42.2                        20                2703
         31-90    88        0.10           53.4                        41                1314
        91-180  6558        0.15           43.7                       104                3434
          181+    17        0.20           23.5                       194                4429

--- Position-controlled check: page_1 tier only ---
n = 7,064 page_1 high-volume pages

freshness_tier    n  median_ctr  pct_under_ctr
          0-30 4519        0.26           45.0
         31-90   28        0.19           57.1
        91-180 2514        0.21           52.5

Observations:
  • The under-CTR rate does not show a strong monotonic increase with staleness.
  • Even within page_1 (controlling

### The rule in plain words

**"A page is worth a CTR review if it has meaningful search visibility (≥500 impressions
in 90 days), sits in an actionable position range (positions 4–20, where a title rewrite
or snippet improvement can plausibly move clicks), and scores higher when it hasn't been
updated recently (stale content = outdated titles/meta). Rank by impression volume
(log-scaled) with a staleness boost, so the biggest missed-click opportunities surface
first."**

**Reason codes** (one per row, priority waterfall):
- `stale_ctr_opportunity` — stale + visible + actionable position
- `fresh_ctr_opportunity` — visible + actionable position, but recently updated
- `no_opportunity` — fails one or more gates

**Action labels:**
- `stale_ctr_opportunity` → `rewrite_title_and_meta`
- `fresh_ctr_opportunity` → `review_serp_snippet`
- `no_opportunity` → `monitor`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

**Score formula** (no fitted weights, fully transparent):
```
visible    = (impressions_90d >= 500)
actionable = (avg_position > 3) AND (avg_position <= 20)
stale      = (days_since_last_update >= 90)
score      = visible × actionable × log1p(impressions_90d) × (1 + stale)
```

- `score = 0` for pages that fail any gate
- Higher impressions → higher score (log-scaled to prevent mega-pages from dominating)
- Stale pages get a 2× boost over fresh ones
- `ctr` is NEVER an input (it's the target proxy)

In [4]:
# ── Build the score ────────────────────────────────────────────────────────

# Gate signals — binary, transparent thresholds
lane4['visible']    = (lane4['impressions_90d'] >= 500).astype(int)
lane4['actionable'] = ((lane4['avg_position'] > 3) & (lane4['avg_position'] <= 20)).astype(int)
lane4['stale']      = (lane4['days_since_last_update'] >= 90).astype(int)

# Score: readable on purpose — no fitted weights
lane4['score'] = (
    lane4['visible']
    * lane4['actionable']
    * np.log1p(lane4['impressions_90d'])
    * (1 + lane4['stale'])
)

# ONE reason code per row (priority waterfall)
def assign_reason(row):
    if row['visible'] and row['actionable'] and row['stale']:
        return 'stale_ctr_opportunity'
    elif row['visible'] and row['actionable']:
        return 'fresh_ctr_opportunity'
    else:
        return 'no_opportunity'

lane4['reason_code'] = lane4.apply(assign_reason, axis=1)

# Action label
action_map = {
    'stale_ctr_opportunity': 'rewrite_title_and_meta',
    'fresh_ctr_opportunity': 'review_serp_snippet',
    'no_opportunity': 'monitor',
}
lane4['action'] = lane4['reason_code'].map(action_map)

# Rank (1 = highest score)
lane4['rank'] = lane4['score'].rank(method='first', ascending=False).astype(int)
lane4 = lane4.sort_values('rank')

# Print summary
print('Score distribution:')
print(f'  Pages with score > 0: {(lane4["score"] > 0).sum():,}')
print(f'  Pages with score = 0: {(lane4["score"] == 0).sum():,}')
print(f'  Max score: {lane4["score"].max():.2f}')
print(f'  Median score (scored pages): {lane4.loc[lane4["score"] > 0, "score"].median():.2f}')
print()
print('Reason code breakdown:')
print(lane4['reason_code'].value_counts().to_string())
print()
print('Action breakdown:')
print(lane4['action'].value_counts().to_string())

Score distribution:
  Pages with score > 0: 11,543
  Pages with score = 0: 10,463
  Max score: 26.31
  Median score (scored pages): 9.18

Reason code breakdown:
reason_code
no_opportunity           10463
fresh_ctr_opportunity     7430
stale_ctr_opportunity     4113

Action breakdown:
action
monitor                   10463
review_serp_snippet        7430
rewrite_title_and_meta     4113


In [5]:
# ── Write the ranked queue CSV ─────────────────────────────────────────────

output_cols = [
    'content_id', 'client_id', 'rank', 'score', 'reason_code', 'action',
    'is_under_ctr', 'impressions_90d', 'avg_position', 'ctr',
    'days_since_last_update', 'position_tier',
]

out = lane4[output_cols].sort_values('rank')

# Ensure the output directory exists
out_dir = pathlib.Path('../../work/outputs')
# Also handle Colab path
if not os.path.exists('../../work'):
    out_dir = pathlib.Path('/content/ShreeyeshAssignment1/work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / 'baseline_action_score.csv'
out.to_csv(csv_path, index=False)
print(f'Wrote {len(out):,} rows to {csv_path}')
print(f'Top-5 preview:')
print(out.head(5).to_string(index=False))

Wrote 22,006 rows to ..\..\work\outputs\baseline_action_score.csv
Top-5 preview:
          content_id         client_id  rank     score           reason_code                 action  is_under_ctr  impressions_90d  avg_position  ctr  days_since_last_update position_tier
content_5fe46e04994d client_4e07408562     1 26.314364 stale_ctr_opportunity rewrite_title_and_meta             1           517715           4.2 0.14                     104        page_1
content_2c2606c5d176 client_19581e27de     2 25.516464 stale_ctr_opportunity rewrite_title_and_meta             0           347399           4.2 0.53                     104        page_1
content_cb112fce36be client_19581e27de     3 25.288081 stale_ctr_opportunity rewrite_title_and_meta             1           309910           5.6 0.16                     104        page_1
content_36ff89c8214e client_19581e27de     4 25.190126 stale_ctr_opportunity rewrite_title_and_meta             1           295097           7.3 0.05                  

In [6]:
# ── Precision@K evaluation ─────────────────────────────────────────────────

def precision_at_k(labels, scores, k):
    """Of the top-K by score, what fraction are truly under-CTR?"""
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

scored = lane4[lane4['score'] > 0]
labels = scored['is_under_ctr'].values
scores = scored['score'].values

p10  = precision_at_k(labels, scores, 10)
p20  = precision_at_k(labels, scores, 20)
p50  = precision_at_k(labels, scores, 50)

print(f'Base rate (under-CTR): {base_rate:.1%}')
print(f'Precision@10:  {p10:.3f}  ({int(p10 * 10)}/10 correct)')
print(f'Precision@20:  {p20:.3f}  ({int(p20 * 20)}/20 correct)')
print(f'Precision@50:  {p50:.3f}  ({int(p50 * 50)}/50 correct)')
print()
if p50 > base_rate:
    print(f'→ The rule beats random ({base_rate:.1%}) at all cutoffs.')
else:
    print(f'→ The rule does NOT beat random at @50. The model must do better.')
print(f'→ A Week-5 model must beat these numbers to earn its complexity.')

Base rate (under-CTR): 46.8%
Precision@10:  0.500  (5/10 correct)
Precision@20:  0.400  (8/20 correct)
Precision@50:  0.340  (17/50 correct)

→ The rule does NOT beat random at @50. The model must do better.
→ A Week-5 model must beat these numbers to earn its complexity.


In [7]:
# ── Write metrics JSON (committable receipt) ───────────────────────────────

metrics = {
    'task': 'Lane 4 — CTR Opportunity Baseline',
    'working_slice_n': int(len(lane4)),
    'scored_n': int((lane4['score'] > 0).sum()),
    'base_rate_under_ctr': round(float(base_rate), 4),
    'precision_at_10': round(float(p10), 4),
    'precision_at_20': round(float(p20), 4),
    'precision_at_50': round(float(p50), 4),
    'score_formula': 'visible * actionable * log1p(impressions_90d) * (1 + stale)',
    'thresholds': {
        'visible': 'impressions_90d >= 500',
        'actionable': '3 < avg_position <= 20',
        'stale': 'days_since_last_update >= 90',
    },
    'reason_codes': ['stale_ctr_opportunity', 'fresh_ctr_opportunity', 'no_opportunity'],
    'signal_verdicts': {
        'ctr_vs_position': 'CONFIRMED',
        'staleness_vs_under_ctr': 'MIXED',
    },
}

json_path = out_dir / 'baseline_metrics.json'
json_path.write_text(json.dumps(metrics, indent=2))
print(f'Wrote metrics to {json_path}')
print(json.dumps(metrics, indent=2))

Wrote metrics to ..\..\work\outputs\baseline_metrics.json
{
  "task": "Lane 4 \u2014 CTR Opportunity Baseline",
  "working_slice_n": 22006,
  "scored_n": 11543,
  "base_rate_under_ctr": 0.4684,
  "precision_at_10": 0.5,
  "precision_at_20": 0.4,
  "precision_at_50": 0.34,
  "score_formula": "visible * actionable * log1p(impressions_90d) * (1 + stale)",
  "thresholds": {
    "visible": "impressions_90d >= 500",
    "actionable": "3 < avg_position <= 20",
    "stale": "days_since_last_update >= 90"
  },
  "reason_codes": [
    "stale_ctr_opportunity",
    "fresh_ctr_opportunity",
    "no_opportunity"
  ],
  "signal_verdicts": {
    "ctr_vs_position": "CONFIRMED",
    "staleness_vs_under_ctr": "MIXED"
  }
}


## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*

In [8]:
# ── Top 10: display the key columns ────────────────────────────────────────

top10 = lane4.head(10)[[
    'rank', 'content_id', 'score', 'reason_code', 'action',
    'is_under_ctr', 'impressions_90d', 'avg_position', 'ctr',
    'days_since_last_update', 'position_tier',
]].copy()

print('Top 10 ranked pages:')
print(top10.to_string(index=False))

Top 10 ranked pages:
 rank           content_id     score           reason_code                 action  is_under_ctr  impressions_90d  avg_position  ctr  days_since_last_update position_tier
    1 content_5fe46e04994d 26.314364 stale_ctr_opportunity rewrite_title_and_meta             1           517715           4.2 0.14                     104        page_1
    2 content_2c2606c5d176 25.516464 stale_ctr_opportunity rewrite_title_and_meta             0           347399           4.2 0.53                     104        page_1
    3 content_cb112fce36be 25.288081 stale_ctr_opportunity rewrite_title_and_meta             1           309910           5.6 0.16                     104        page_1
    4 content_36ff89c8214e 25.190126 stale_ctr_opportunity rewrite_title_and_meta             1           295097           7.3 0.05                     104        page_1
    5 content_c21024970297 24.522702 stale_ctr_opportunity rewrite_title_and_meta             0           211366           5.1 0.

In [9]:
# ── Top-10 skeptic review: one line each ───────────────────────────────────
# For each: action, why it's there, what would make it wrong.

print('Top-10 Skeptic Review')
print('=' * 90)

for _, row in lane4.head(10).iterrows():
    r = int(row['rank'])
    cid = row['content_id'][-8:]  # last 8 chars for brevity
    act = row['action']
    reason = row['reason_code']
    imp = int(row['impressions_90d'])
    pos = row['avg_position']
    ctr_val = row['ctr']
    days = int(row['days_since_last_update'])
    under = 'YES' if row['is_under_ctr'] else 'NO'
    tier = row['position_tier']

    print(f'\nRank {r} — ...{cid}  (under-CTR: {under})')
    print(f'  Action: {act}')
    print(f'  Why: {reason} — {imp:,} impressions, position {pos:.1f} ({tier}), '
          f'{days}d since update, CTR {ctr_val:.2f}%')

    # What would make it wrong — specific to each page's characteristics
    wrong_reasons = []
    if row['is_under_ctr'] == 0:
        wrong_reasons.append(
            f'ALREADY WRONG: this page is NOT under its tier median CTR — '
            f'the rule flagged it on volume + position + staleness alone')
    if imp > 50000:
        wrong_reasons.append(
            'mega-volume page may be a branded query where CTR is naturally '
            'position-driven and title changes won\'t help')
    if pos <= 5:
        wrong_reasons.append(
            'position 3-5 pages may already have optimal CTR; '
            'improvement might require ranking higher, not rewriting')
    if days < 30:
        wrong_reasons.append(
            'recently updated — stale flag shouldn\'t apply, but fresh_ctr '
            'opportunity may still be noise from a recent content change')
    if ctr_val > 0.3 and tier in ('page_1', 'striking'):
        wrong_reasons.append(
            f'CTR {ctr_val:.2f}% is already above the Q25 for {tier}; '
            'the gap may be too small to justify review effort')
    if not wrong_reasons:
        wrong_reasons.append(
            'could be wrong if the low CTR is driven by SERP features '
            '(featured snippets, knowledge panels) that no title change can fix')

    for wr in wrong_reasons:
        print(f'  Wrong if: {wr}')

print(f'\n{"=" * 90}')
correct_in_top10 = lane4.head(10)['is_under_ctr'].sum()
print(f'Summary: {correct_in_top10}/10 top-ranked pages are genuinely under their tier CTR.')
if correct_in_top10 < 10:
    print(f'  → {10 - correct_in_top10} false positive(s) — the rule over-flags on volume + staleness.')
    print('  → The model must learn to distinguish these from true under-performers.')

Top-10 Skeptic Review

Rank 1 — ...6e04994d  (under-CTR: YES)
  Action: rewrite_title_and_meta
  Why: stale_ctr_opportunity — 517,715 impressions, position 4.2 (page_1), 104d since update, CTR 0.14%
  Wrong if: mega-volume page may be a branded query where CTR is naturally position-driven and title changes won't help
  Wrong if: position 3-5 pages may already have optimal CTR; improvement might require ranking higher, not rewriting

Rank 2 — ...06c5d176  (under-CTR: NO)
  Action: rewrite_title_and_meta
  Why: stale_ctr_opportunity — 347,399 impressions, position 4.2 (page_1), 104d since update, CTR 0.53%
  Wrong if: ALREADY WRONG: this page is NOT under its tier median CTR — the rule flagged it on volume + position + staleness alone
  Wrong if: mega-volume page may be a branded query where CTR is naturally position-driven and title changes won't help
  Wrong if: position 3-5 pages may already have optimal CTR; improvement might require ranking higher, not rewriting
  Wrong if: CTR 0.53

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# ── Weak picks analysis ────────────────────────────────────────────────────

top20 = lane4.head(20)
false_positives = top20[top20['is_under_ctr'] == 0]

print(f'Weak picks in the top 20:')
print(f'  {len(false_positives)} of 20 are NOT under their tier median CTR (false positives)\n')

if len(false_positives) > 0:
    print('These pages scored high because of volume + position + staleness,')
    print('but their CTR is actually at or above the tier median:\n')
    print(false_positives[[
        'rank', 'content_id', 'impressions_90d', 'avg_position',
        'ctr', 'tier_median_ctr', 'days_since_last_update', 'position_tier',
    ]].to_string(index=False))
    print()
    print('Why they\'re weak:')
    print('  • The rule has no way to know the page\'s CTR — it only sees volume,')
    print('    position, and staleness. High-volume stale pages that happen to have')
    print('    GOOD CTR get ranked just as high as those with bad CTR.')
    print('  • This is exactly the gap a model can close: it can learn engagement,')
    print('    content type, and word count patterns that distinguish true under-')
    print('    performers from pages that are fine despite being stale.')
else:
    print('All top-20 picks are genuinely under-CTR. Look harder at top 50...')
    top50 = lane4.head(50)
    fp50 = top50[top50['is_under_ctr'] == 0]
    print(f'  In the top 50: {len(fp50)} false positives')
    if len(fp50) > 0:
        print(fp50[[
            'rank', 'content_id', 'impressions_90d', 'avg_position',
            'ctr', 'tier_median_ctr', 'days_since_last_update',
        ]].head(5).to_string(index=False))

Weak picks in the top 20:
  12 of 20 are NOT under their tier median CTR (false positives)

These pages scored high because of volume + position + staleness,
but their CTR is actually at or above the tier median:

 rank           content_id  impressions_90d  avg_position  ctr  tier_median_ctr  days_since_last_update position_tier
    2 content_2c2606c5d176           347399           4.2 0.53             0.23                     104        page_1
    5 content_c21024970297           211366           5.1 0.41             0.23                     104        page_1
    7 content_d17681677e69           201584           5.8 0.24             0.23                     104        page_1
    9 content_c5063073d048           192205          12.5 0.24             0.15                     104      striking
   10 content_3d94572c3a35           190623           4.3 0.24             0.23                     104        page_1
   11 content_01908772c6db           187893           4.0 0.45             0.2

In [11]:
# ── Leakage check ─────────────────────────────────────────────────────────

print('Leakage verification')
print('=' * 60)

# The score formula uses ONLY these inputs:
score_inputs = ['impressions_90d', 'avg_position', 'days_since_last_update']

# These must NEVER be score inputs:
forbidden = ['ctr', 'trend_direction', 'trend_pct', 'is_declining_label',
             'ctr_gap', 'is_under_ctr', 'tier_median_ctr']

print(f'Score inputs: {score_inputs}')
print(f'Forbidden columns (label-derived or target proxy):')
for col in forbidden:
    used = col in score_inputs
    status = '✗ LEAKED' if used else '✓ not used'
    print(f'  {col}: {status}')

print()
print('Future-window check:')
print('  • impressions_90d: trailing 90-day window (past) ✓')
print('  • avg_position: trailing 90-day average (past) ✓')
print('  • days_since_last_update: content metadata (past) ✓')
print('  • No forward-looking columns used.')
print()
print('Verdict: NO LEAKAGE DETECTED.')
print('  The score uses only backward-looking, observable signals.')
print('  CTR (the target proxy) is never an input to the score.')

Leakage verification
Score inputs: ['impressions_90d', 'avg_position', 'days_since_last_update']
Forbidden columns (label-derived or target proxy):
  ctr: ✓ not used
  trend_direction: ✓ not used
  trend_pct: ✓ not used
  is_declining_label: ✓ not used
  ctr_gap: ✓ not used
  is_under_ctr: ✓ not used
  tier_median_ctr: ✓ not used

Future-window check:
  • impressions_90d: trailing 90-day window (past) ✓
  • avg_position: trailing 90-day average (past) ✓
  • days_since_last_update: content metadata (past) ✓
  • No forward-looking columns used.

Verdict: NO LEAKAGE DETECTED.
  The score uses only backward-looking, observable signals.
  CTR (the target proxy) is never an input to the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with visible bucket tables and n (at least one flag-linked) ✓
- [x] One rule with a score, a reason code, and an action label ✓
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv` ✓
- [x] Top-10 reviewed with "what would make it wrong" for each ✓
- [x] No future-window or label-derived inputs in the score ✓
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.